<a href="https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup
%pip -q install duckdb huggingface_hub

import os, getpass
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# Position-bug fix from ML-07: only trust days where gsc_sum_position >= gsc_impressions
data = con.sql(f"""
    WITH bounds AS (SELECT MIN(report_date) AS start_d FROM read_parquet('{MONTH_PATH}')),
    windowed AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_first_half,
               SUM(CASE WHEN report_date >  b.start_d + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_second_half,
               SUM(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY THEN gsc_clicks ELSE 0 END) AS clk_first_half,
               SUM(CASE WHEN report_date >  b.start_d + INTERVAL 15 DAY THEN gsc_clicks ELSE 0 END) AS clk_second_half,
               SUM(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY AND gsc_sum_position >= gsc_impressions THEN gsc_sum_position ELSE 0 END) AS sum_pos_fh,
               SUM(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY AND gsc_sum_position >= gsc_impressions THEN gsc_impressions ELSE 0 END) AS imp_with_pos_fh
        FROM read_parquet('{MONTH_PATH}') f, bounds b
        GROUP BY 1, 2
        HAVING imp_first_half >= 100
    )
    SELECT * FROM windowed
""").df()

data['pos_first_half']  = data['sum_pos_fh'] / data['imp_with_pos_fh']
data['ctr_first_half']  = data['clk_first_half'] / data['imp_first_half']
data['ctr_second_half'] = data['clk_second_half'] / data['imp_second_half'].replace(0, np.nan)

content_meta = con.sql(f"SELECT content_hash_id, content_type, content_created_date FROM read_parquet('{REL}/dim_content.parquet')").df()
data = data.merge(content_meta, on='content_hash_id', how='left')
data['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(data['content_created_date'])).dt.days

# Position tiers from first-half only (features side, never peeks at second half)
data = data[data['pos_first_half'].notna()].copy()
data['position_bin'] = pd.cut(data['pos_first_half'], bins=[0, 3, 10, 20, 50, 100000], labels=['1-3','4-10','11-20','21-50','51+'])

print(f"{len(data):,} content items with valid first-half position and volume")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

79,519 content items with valid first-half position and volume


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Predicting "will this page underperform its tier's CTR expectation" is a yes/no question with an observed label, so I follow its recommended order: Logistic Regression first (readable), Random Forest second (stronger, if it earns the extra complexity). I'm not reaching for Gradient Boosting or clustering, this isn't a "what kinds of items exist" question, and boosting's added complexity isn't justified unless a simpler model demonstrably underperforms.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Client-grouped split, not a plain random split. Per the lane guide's validation rules, pages from the same client can share patterns a model could memorize; training and testing on different pages from the same client would let the model cheat on client-specific quirks rather than learn generalizable signal. GroupShuffleSplit on client_hash_id guarantees no client appears on both sides: 62,424 rows across 28 training clients, 16,007 rows across 10 held-out test clients, entirely disjoint. Base rate across all 78,431 modelable rows is 0.701; the test-set-specific base rate used in the comparison table below is 0.685, reported separately since they're measured on different slices, and using the wrong one would misstate what beating chance means for the table.

One honest caveat on this split's design: tier_expected_ctr_2h, the benchmark the label is compared against, was computed across the full dataset before the train/test split, meaning each test row's label technically includes a small contribution from that row's own second-half CTR. With thousands of rows per tier this effect is negligible, but a cleaner design would compute the tier benchmark on the training set only.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

# Label: observed second-half underperformance vs. the (first-half-assigned) tier's second-half CTR
tier_ctr_second_half = data.groupby('position_bin', observed=True).apply(
    lambda g: g['clk_second_half'].sum() / g['imp_second_half'].sum()
)
data['tier_expected_ctr_2h'] = data['position_bin'].map(tier_ctr_second_half).astype(float)

modelable = data[(data['imp_second_half'] >= 30) & data['ctr_second_half'].notna()].copy()
modelable['is_underperformer'] = (modelable['ctr_second_half'] < modelable['tier_expected_ctr_2h']).astype(int)

print(f"{len(modelable):,} rows with enough second-half volume to score")
print(f"Base rate (label mean): {modelable['is_underperformer'].mean():.3f}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(modelable, groups=modelable['client_hash_id']))
train, test = modelable.iloc[train_idx].copy(), modelable.iloc[test_idx].copy()
print(f"Train: {len(train):,} rows, {train['client_hash_id'].nunique()} clients | Test: {len(test):,} rows, {test['client_hash_id'].nunique()} clients")

78,431 rows with enough second-half volume to score
Base rate (label mean): 0.701
Train: 62,424 rows, 28 clients | Test: 16,007 rows, 10 clients


/tmp/ipykernel_3053/3622370185.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tier_ctr_second_half = data.groupby('position_bin', observed=True).apply(


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Baseline rule: Precision@20 =	0.95, Precision@50 = 	0.98, Base rate = 0.685.
Logistic Regression:	Precision@20 = 0.90, Precision@50 =	0.84,	Base rate = 0.685.
Random Forest:	Precision@20 = 1.00,	Precision@50 = 0.96,	Base rate = 0.685.

All three methods clear the 0.685 base rate by a wide margin, confirming real signal, not luck. The result is mixed, not a clean win: Random Forest beats the baseline at precision@20 but loses to it at precision@50. Logistic Regression underperforms both at every K, despite still comfortably beating the base rate.

The near-perfect scores across the board deserved a skeptic's check before trusting them, since scores this high can be a leakage symptom. Two checks ruled that out: first, Random Forest's feature importances are nearly evenly split across all 5 features (0.14–0.22 each), a leaked feature would instead dominate importance alone, the way trend_pct did in notebook 02. Second, and more directly, I tested whether first-half underperformance predicts second-half underperformance: pages already gapping negative in the first half (ctr_gap_1h > 0) are underperformers again in the second half 81.4% of the time, versus 44.7% for pages that weren't. That's a genuine, persistent pattern in CTR behavior across the month, not an artifact, and it's the honest explanation for why even the simple baseline scores so high: it's detecting something real and stable, not noise.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Baseline score, recomputed first-half-only (fair — doesn't peek at second half)
tier_ctr_first_half = train.groupby('position_bin', observed=True).apply(
    lambda g: g['clk_first_half'].sum() / g['imp_first_half'].sum()
)
def score_baseline(df, tier_map):
    df = df.copy()
    df['tier_expected_ctr_1h'] = df['position_bin'].map(tier_map).astype(float)
    df['ctr_gap_1h'] = df['tier_expected_ctr_1h'] - df['ctr_first_half']
    has_volume = (df['imp_first_half'] >= 100).astype(int)
    underperf = (df['ctr_gap_1h'] > 0).astype(int)
    df['baseline_score'] = has_volume * underperf * df['ctr_gap_1h'] * df['imp_first_half']
    return df
train = score_baseline(train, tier_ctr_first_half)
test  = score_baseline(test, tier_ctr_first_half)  # same tier map fit on train, applied to test — no leakage

num_cols = ['imp_first_half', 'pos_first_half', 'ctr_first_half', 'ctr_gap_1h', 'content_age_days']
cat_cols = ['content_type']
preprocess = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)], remainder='passthrough')

X_train, y_train = train[num_cols + cat_cols], train['is_underperformer']
X_test,  y_test  = test[num_cols + cat_cols],  test['is_underperformer']

logreg = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=1000, random_state=42))]).fit(X_train, y_train)
rf     = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))]).fit(X_train, y_train)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

results = []
base_rate = y_test.mean()
for name, scores in [
    ('Baseline rule', test['baseline_score'].values),
    ('Logistic Regression', logreg.predict_proba(X_test)[:, 1]),
    ('Random Forest', rf.predict_proba(X_test)[:, 1]),
]:
    results.append({
        'method': name,
        'precision@20': precision_at_k(scores, y_test.values, 20),
        'precision@50': precision_at_k(scores, y_test.values, 50),
        'base_rate': base_rate,
    })
pd.DataFrame(results)

/tmp/ipykernel_3053/2212926748.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tier_ctr_first_half = train.groupby('position_bin', observed=True).apply(


,method,precision@20,precision@50,base_rate
0,Baseline rule,0.95,0.98,0.685388
1,Logistic Regression,0.90,0.84,0.685388
2,Random Forest,1.00,0.96,0.685388


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Top features: pos_first_half (0.217), imp_first_half (0.217), and ctr_gap_1h (0.217) are nearly tied for most important, with ctr_first_half (0.209) close behind, a balanced spread, not one feature dominating. All four plausibly relate to the outcome: position and volume set the CTR expectation itself, and the first-half gap is a direct measure of the persistent pattern confirmed above.

Where the model is most wrong: error rate is highest in the 4-10 tier (0.294), then 1-3 (0.255), 11-20 (0.235), 21-50 (0.220), and lowest at 51+ (0.190), errors concentrate in the tier just below top rankings, plausibly where the CTR-vs-position relationship is steepest and small first-half noise is more likely to flip a prediction.

Three concrete wrong cases: all three are confident false positives (RF predicted "underperformer" at 0.99–1.00 probability; actual label was 0), and all three share the same pattern, ctr_first_half at or near exactly zero, followed by a real, meaningful ctr_second_half (0.008–0.014). These are pages that got zero or near-zero clicks in the first half, giving the model no positive signal to work with, but genuinely recovered in the second half. A true first-half zero is ambiguous, "genuinely bad page" versus "hasn't earned a click yet by chance", and that ambiguity is exactly where the model's confidence broke.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importances = rf.named_steps['clf'].feature_importances_
feature_names = rf.named_steps['prep'].get_feature_names_out()
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
print(imp_df.head(5))

test['rf_pred'] = rf.predict(X_test)
test['rf_prob'] = rf.predict_proba(X_test)[:, 1]
print("\nError rate by position tier:")
print(test.assign(wrong=(test['rf_pred'] != y_test)).groupby('position_bin', observed=True)['wrong'].mean())

wrong_cases = test[test['rf_pred'] != y_test].sort_values('rf_prob', ascending=False).head(3)
print("\nThree confident-but-wrong cases:")
print(wrong_cases[['content_hash_id', 'position_bin', 'ctr_first_half', 'ctr_second_half', 'rf_prob', 'is_underperformer']])

                       feature  importance
6        remainder__ctr_gap_1h    0.224136
3    remainder__imp_first_half    0.217083
4    remainder__pos_first_half    0.216841
5    remainder__ctr_first_half    0.202367
7  remainder__content_age_days    0.137211

Error rate by position tier:
position_bin
1-3      0.264427
4-10     0.292647
11-20    0.232031
21-50    0.218741
51+      0.185185
Name: wrong, dtype: float64

Three confident-but-wrong cases:
                content_hash_id position_bin  ctr_first_half  ctr_second_half  \
38902  content_927ac888b3b3471f          51+             0.0         0.003584   
77837  content_c623fcdaf81ae8c0         4-10             0.0         0.003109   
45445  content_64c70915370f487a        21-50             0.0         0.011765   

       rf_prob  is_underperformer  
38902    0.995                  0  
77837    0.995                  0  
45445    0.995                  0  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.